# 🎵 AICoverGen WebUI — Kaggle versiyasi (O'zbekcha)
Ushbu notebook **AICoverGen** dasturini **Kaggle** platformasida bepul GPU (T4 yoki P100) yordamida xatosiz ishga tushirish uchun maxsus moslashtirilgan.

---
### ⚠️ ISHGA TUSHIRISH OLDIDAN SHU 2 TA SOZLAMANI TEKSHIRING (O'ng tarafdagi menyu):
1. **Internet**: `Settings` -> `Internet` -> **ON (Yoqilgan)** bo'lishi SHART! *(Agar sizda yoqilmasa, telefon raqamingiz orqali Kaggle hisobingizni tasdiqlang)*
2. **Accelerator**: `Settings` -> `Accelerator` -> **GPU T4 x2** yoki **GPU P100** ni tanlang.

---
### 🚀 Bitta bosishda ishga tushirish:
Quyidagi katakchaning `Play (▶)` tugmasini bosing. Dastur o'zi avtomatik barcha kutubxonalarni o'rnatadi, modellarni yuklaydi va sizga WebUI ga kirish uchun to'q sariq **CLOUDFLARE** silkasini chiqarib beradi!

In [ ]:
# =====================================================================
# 🎵 AICoverGen WebUI — Kaggle (Bitta bosishda to'liq ishga tushirish)
# =====================================================================
import os, sys, shutil, re, subprocess, time
from IPython.display import HTML, display, clear_output

print("🔄 1/4 Tizim paketlari o'rnatilmoqda (sox, ffmpeg, cloudflared)...")
!apt-get update -y -qq > /dev/null
!apt-get install -y -qq sox libsox-dev libsox-fmt-all ffmpeg curl wget > /dev/null

# Cloudflared tunnelni yuklab olish
if not os.path.exists("/usr/local/bin/cloudflared"):
    !wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod +x /usr/local/bin/cloudflared

print("🔄 2/4 Kutubxonalar sozlanmoqda (Numpy 1.26, Cython, Fairseq)...")
!pip install -q --no-warn-conflicts "pip<24.1"
!pip install -q --no-warn-conflicts "cython<3.0.0" "numpy==1.26.4" hydra-core==1.0.7 omegaconf==2.0.6

WORKDIR = "/kaggle/working"
FAIRSEQ_DIR = os.path.join(WORKDIR, "fairseq")
if not os.path.exists(FAIRSEQ_DIR):
    !git clone --depth 1 https://github.com/pytorch/fairseq.git {FAIRSEQ_DIR}

# Fairseq OmegaConf va Python moslik patchlari
for fpath in [os.path.join(FAIRSEQ_DIR, "fairseq/__init__.py"), os.path.join(FAIRSEQ_DIR, "fairseq/dataclass/initialize.py")]:
    if os.path.exists(fpath):
        with open(fpath, 'r', encoding='utf-8') as f: c = f.read()
        c = c.replace('hydra_init()', '# hydra_init()')
        c = re.sub(r'def hydra_init\([^)]*\)[^:]*:', 'def hydra_init(*args, **kwargs):\n    return', c)
        with open(fpath, 'w', encoding='utf-8') as f: f.write(c)

!pip install -q --no-warn-conflicts --no-build-isolation --editable {FAIRSEQ_DIR}
!pip install -q --no-warn-conflicts pedalboard pydub pyworld praat-parselmouth faiss-cpu yt-dlp torchcrepe sox soundfile librosa deemix tqdm pyngrok onnxruntime-gpu gradio-client==0.8.1 "gradio==3.50.2" "fastapi<0.110" "starlette<0.37"

# Gradio frpc binar fayli
try:
    import gradio, urllib.request
    gradio_dir = os.path.dirname(gradio.__file__)
    frpc_path = os.path.join(gradio_dir, "frpc_linux_amd64_v0.2")
    if not os.path.exists(frpc_path):
        urllib.request.urlretrieve("https://cdn-media.huggingface.co/frpc-gradio-0.2/frpc_linux_amd64", frpc_path)
        os.chmod(frpc_path, 0o755)
except Exception:
    pass

print("🔄 3/4 AICoverGen repozitoriysi va tayyor AI modellar yuklanmoqda...")
REPO_DIR = os.path.join(WORKDIR, "HRVC")
if not os.path.exists(os.path.join(REPO_DIR, "src/webui.py")):
    if os.path.exists(REPO_DIR):
        shutil.rmtree(REPO_DIR)
    !git clone https://github.com/nidayboy/aicovergenN.git {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git fetch origin main && git reset --hard origin/main

%cd {REPO_DIR}
!python src/download_models.py

print("🔄 4/4 WebUI va Cloudflare tunnel ishga tushirilmoqda...")
PORT = 7860
!fuser -k 7860/tcp 2>/dev/null || true
!pkill -9 -f "webui.py" 2>/dev/null || true
!pkill -9 -f "cloudflared" 2>/dev/null || true
os.environ["TENSORBOARD_NO_TF"] = "1"

cf_log = "/kaggle/working/cloudflared.log"
if os.path.exists(cf_log): os.remove(cf_log)
!nohup cloudflared tunnel --url http://127.0.0.1:{PORT} > {cf_log} 2>&1 &

cloudflared_url = None
print("⏳ Cloudflare tunnel tayyorlanmoqda...")
for _ in range(25):
    time.sleep(1)
    if os.path.exists(cf_log):
        with open(cf_log, 'r', encoding='utf-8', errors='ignore') as f:
            content = f.read()
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', content)
        if match:
            cloudflared_url = match.group(0)
            break

cmd = ["python", "-u", "src/webui.py", "--listen", "--listen-port", str(PORT), "--share"]
process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    universal_newlines=True
)

banner_shown = False
def show_banner(cf_link, gr_link=None):
    buttons = ""
    if cf_link:
        buttons += f'''<a href="{cf_link}" target="_blank" style="background: #f38020; color: white; padding: 14px 28px; text-decoration: none; border-radius: 10px; font-weight: bold; font-size: 18px; margin: 8px; display: inline-block; box-shadow: 0 4px 12px rgba(243,128,32,0.4);">⚡ AICoverGen Kirish (Cloudflare)</a>'''
    if gr_link:
        buttons += f'''<a href="{gr_link}" target="_blank" style="background: #ff5722; color: white; padding: 14px 28px; text-decoration: none; border-radius: 10px; font-weight: bold; font-size: 18px; margin: 8px; display: inline-block;">🌐 Gradio Kirish</a>'''
    
    html = f'''
    <div style="background: #181824; border: 2px solid #6366f1; border-radius: 14px; padding: 25px; text-align: center; margin: 15px 0;">
        <h2 style="color: #ffffff; margin-top: 0; font-size: 24px;">🎉 AICoverGen WebUI Muvaffaqiyatli Ishga Tushdi!</h2>
        <p style="color: #cbd5e1; font-size: 15px;">WebUI dasturiga kirish uchun quyidagi tugmani bosing:</p>
        <div style="margin: 20px 0;">{buttons}</div>
        <div style="color: #94a3b8; font-size: 13px;">
            {f'<p>• <b>Cloudflare havola:</b> <a href="{cf_link}" target="_blank" style="color: #fb923c;">{cf_link}</a></p>' if cf_link else ''}
            {f'<p>• <b>Gradio havola:</b> <a href="{gr_link}" target="_blank" style="color: #ff7043;">{gr_link}</a></p>' if gr_link else ''}
        </div>
    </div>
    '''
    display(HTML(html))

if cloudflared_url:
    show_banner(cloudflared_url)
    banner_shown = True

for line in iter(process.stdout.readline, ''):
    print(line, end='', flush=True)
    if 'Running on public URL:' in line:
        match = re.search(r'https://[a-zA-Z0-9-]+\.gradio\.live', line)
        if match:
            show_banner(cloudflared_url, match.group(0))
            banner_shown = True
